# Replicant — Convergence Benchmark Analysis (CSV)

Loads one of `results/results{,-docker,-k8s}.csv` and produces thesis-ready figures in `figures/<source>/`. Pick which result set to analyse via the `SOURCE` constant in the next cell.

**Generate results first** (release build — debug timings carry tens of ms of overhead that swamps real signal):

| `SOURCE`     | How to generate                                                                                       | CSV written to                  |
| ------------ | ----------------------------------------------------------------------------------------------------- | ------------------------------- |
| `in_process` | `cargo run --release --bin orchestrator -- --trials 10 --output csv scenarios/*.toml > results/results.csv` | `results/results.csv`           |
| `docker`     | `just bench-docker "scenarios/*.toml" 10`                                                              | `results/results-docker.csv`    |
| `k8s`        | `just bench-k8s "scenarios/*.toml" 10`                                                                 | `results/results-k8s.csv`       |

The `just bench-*` recipes own the stack/cluster lifecycle and redirect the orchestrator's CSV themselves; the in-process flow uses your shell's `>` redirect.

Companion notebooks:
- [`protocol_metrics.ipynb`](protocol_metrics.ipynb) — OTel JSON ingestion (sync traffic, op latency, doc size). `in_process` only.
- [`live_metrics.ipynb`](live_metrics.ipynb) — live PromQL queries against a running docker/k8s stack.
- [`comparison.ipynb`](comparison.ipynb) — cross-source comparison; does the topology ranking survive the deployment switch?

> **Cache:** Each CSV is cached alongside it as `<stem>.parquet` for faster reruns. The cache is refreshed automatically when the CSV is newer than the parquet. To force a rebuild, delete the parquet file.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

# Pick which result set the notebook analyses. Switch this single value and
# rerun to produce a figure set for that deployment target.
#   in_process: results/results.csv         (in-process orchestrator runs)
#   docker:     results/results-docker.csv  (from `just bench-docker`)
#   k8s:        results/results-k8s.csv     (from `just bench-k8s`)
SOURCE = "k8s"

REPO    = Path("..").resolve()  # absolute repo root; works regardless of launch CWD
RESULTS = REPO / "results"
CSV     = RESULTS / (
    "results.csv" if SOURCE == "in_process" else f"results-{SOURCE}.csv"
)
PARQUET = CSV.with_suffix(".parquet")
# Per-source subdir so figures from different deployment targets don't
# overwrite each other (e.g. when comparing docker vs k8s side-by-side).
FIGS    = REPO / "analysis" / "figures" / SOURCE
FIGS.mkdir(parents=True, exist_ok=True)

## Load data

On first run the CSV is parsed and cached as Parquet (preserves dtypes, loads faster on reruns).

In [ ]:
csv_mtime = CSV.stat().st_mtime if CSV.exists() else 0
parquet_fresh = PARQUET.exists() and PARQUET.stat().st_mtime >= csv_mtime

if parquet_fresh:
    df = pd.read_parquet(PARQUET)
else:
    df = pd.read_csv(CSV)
    df.to_parquet(PARQUET)

trials  = df[df.row_type == "trial"].copy()
summary = df[df.row_type == "summary"].copy()
# In summary rows the `trial` column holds the trial count, not a trial number.
summary = summary.rename(columns={"trial": "n_trials"})

print(f"{len(trials)} trial records, {trials.scenario.nunique()} scenarios, "
      f"{trials.groupby('scenario').size().iloc[0]} trials each")
summary[["scenario", "n_trials", "node_count", "op_count", "mean_ms", "p50_ms", "p95_ms"]]

## Partition-heal: convergence vs node count

Measures time from heal trigger (cross-group edges added) to global convergence.
`ConnectPeer` blocks until the Automerge sync handshake is open (via an internal
readiness signal), so these numbers reflect actual CRDT merge cost rather than
any fixed settle delay.

Split by **two** dimensions:
- **`write_pattern`** (`round_robin` vs `concentrated`) — how writes were distributed during the partition phase.
- **`heal_topology`** (`full_mesh` vs `bridge`) — what wiring is added on heal. `full_mesh` reconnects every pair across the two groups; `bridge` adds only `groups[0].nodes[0] ↔ groups[1].nodes[0]`, forcing all cross-partition state through one edge.

The bridge variant is where the thesis story is loudest: with one cross-edge it has a longer diameter (3 hops) but vastly fewer edges to flood (3-13 vs 6-28 in full-mesh-heal). Convergence is **~7-21× faster** than full-mesh heal — relay amplification dominates over diameter even on the heal path. This is the same mechanism that makes line-n10 beat full-mesh-n10 in the steady-state experiments, applied to partition recovery.

In [ ]:
heal = summary[summary.scenario.str.startswith("partition")].copy()
heal["write_pattern"] = heal.scenario.apply(
    lambda s: "concentrated" if "concentrated" in s else "round_robin"
)
heal["heal_topology"] = heal.scenario.apply(
    lambda s: "bridge" if "bridge" in s else "full_mesh"
)
heal["node_count"] = heal.node_count.astype(int)
heal = heal.sort_values(["node_count", "heal_topology", "write_pattern"])

node_counts = sorted(heal.node_count.unique())
# Series order keeps full_mesh on the left of each cluster so the bridge
# speedup reads visually as a step down between the two groups.
series = [
    ("full_mesh", "round_robin"),
    ("full_mesh", "concentrated"),
    ("bridge", "round_robin"),
    ("bridge", "concentrated"),
]
n_series = len(series)
width = 0.78 / n_series

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (heal_top, pat) in enumerate(series):
    sub = (
        heal[(heal.heal_topology == heal_top) & (heal.write_pattern == pat)]
        .set_index("node_count")
        .reindex(node_counts)
    )
    offsets = [n + (i - (n_series - 1) / 2) * width for n in node_counts]
    yerr = (sub.p95_ms - sub.mean_ms).clip(lower=0).fillna(0)
    ax.bar(
        offsets,
        sub.mean_ms.fillna(0),
        width=width,
        yerr=yerr,
        capsize=3,
        label=f"{heal_top} / {pat}",
        alpha=0.85,
    )
ax.set_xticks(node_counts)
ax.set_xticklabels([f"n={n}" for n in node_counts])
ax.set_xlabel("Total nodes (2 partitions)")
ax.set_ylabel("Heal convergence (ms)")
ax.set_title("Partition-heal: convergence by heal_topology × write_pattern")
ax.legend(title="heal_topology / write_pattern", loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(FIGS / "partition_heal.pdf")
plt.show()

## Convergence vs N — by topology kind × write pattern

Each topology family scales differently, and within a family, write pattern matters: `concentrated` (all ops at `node-0`, dashed) vs `round_robin` (writes distributed across all nodes, solid). Error bars span p50–p95.

Concentrated tends to win within a topology: with a single author, replicas integrate changes in the same order, so peer-by-peer sync converges in a single wave instead of reconciling the divergence that scattered writes create. The topology gap (sparse line/ring vs dense full-mesh) is typically larger than the write-pattern gap — relay amplification dominates.

In [ ]:
import matplotlib.patches as mpatches

topo = summary[summary.topology_kind.isin(["full_mesh", "ring", "line", "star"])].copy()
topo["write_pattern"] = topo.scenario.apply(
    lambda s: "concentrated" if s.endswith("-concentrated") else "round_robin"
)
topo = topo.sort_values(["topology_kind", "write_pattern", "node_count"])

kinds = sorted(topo.topology_kind.unique())
palette = dict(zip(kinds, sns.color_palette("muted", n_colors=len(kinds))))
linestyles = {"round_robin": "-", "concentrated": "--"}

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for (kind, pat), group in topo.groupby(["topology_kind", "write_pattern"]):
    yerr_low = (group.mean_ms - group.p50_ms).clip(lower=0)
    yerr_high = (group.p95_ms - group.mean_ms).clip(lower=0)
    ax.errorbar(
        group.node_count,
        group.mean_ms,
        yerr=[yerr_low, yerr_high],
        marker="o",
        capsize=3,
        color=palette[kind],
        linestyle=linestyles[pat],
    )

color_legend = [mpatches.Patch(color=palette[k], label=k) for k in kinds]
style_legend = [
    plt.Line2D([0], [0], color="black", linestyle=linestyles[p], label=p)
    for p in linestyles
]
first_legend = ax.legend(handles=color_legend, title="Topology",
                          loc="upper left", bbox_to_anchor=(1.02, 1))
ax.add_artist(first_legend)
ax.legend(handles=style_legend, title="Write pattern",
          loc="upper left", bbox_to_anchor=(1.02, 0.55))

ax.set_xlabel("Node count")
ax.set_ylabel("Convergence (ms)")
ax.set_title("Convergence vs N — by topology × write pattern")
fig.tight_layout()
fig.savefig(FIGS / "convergence_vs_n_by_topology.pdf", bbox_inches="tight")
plt.show()

## Convergence vs diameter — the structural correlate

Diameter is the longest shortest-path in the topology — the minimum number of hops any state must traverse to fully propagate. Conventional wisdom predicts `convergence_ms ~ diameter`, but Phase A's relay-on-receive amplifies with **edge count** and **max degree** too.

Below scatters every non-partition scenario by `(diameter, mean_ms)`, coloured by topology kind and shaped by write pattern. **Diameter alone does not predict convergence**: full-mesh (diameter = 1) ranges from <1 ms (n = 2) to >40 ms (n = 10) because edge count grows with N²; line-n10 (diameter 9) converges faster than full-mesh-n10 (diameter 1) because the line has only 9 edges vs 45.

This is the thesis's negative result: diameter is necessary (no traversal can be shorter) but not sufficient for predicting CRDT convergence under a relay-flooded sync layer.

In [ ]:
import matplotlib.patches as mpatches

diam_df = summary[summary.topology_kind != "partition_heal"].copy()
diam_df["write_pattern"] = diam_df.scenario.apply(
    lambda s: "concentrated" if s.endswith("-concentrated") else "round_robin"
)

kinds = sorted(diam_df.topology_kind.unique())
palette = dict(zip(kinds, sns.color_palette("muted", n_colors=len(kinds))))
markers = {"round_robin": "o", "concentrated": "s"}

fig, ax = plt.subplots(figsize=(8, 5))
for (kind, pat), group in diam_df.groupby(["topology_kind", "write_pattern"]):
    ax.scatter(
        group.diameter,
        group.mean_ms,
        marker=markers[pat],
        s=90,
        color=palette[kind],
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
    )

for _, row in diam_df.iterrows():
    ax.annotate(
        f"n={row.node_count}",
        (row.diameter, row.mean_ms),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=8,
        color="dimgray",
    )

color_legend = [mpatches.Patch(color=palette[k], label=k) for k in kinds]
shape_legend = [
    plt.Line2D([0], [0], marker=markers[pat], color="black", linestyle="",
               markerfacecolor="lightgray", markersize=10, label=pat)
    for pat in markers
]
first_legend = ax.legend(handles=color_legend, title="Topology",
                          loc="upper left", bbox_to_anchor=(1.02, 1))
ax.add_artist(first_legend)
ax.legend(handles=shape_legend, title="Write pattern",
          loc="upper left", bbox_to_anchor=(1.02, 0.55))

ax.set_xlabel("Diameter (hops)")
ax.set_ylabel("Convergence (ms)")
ax.set_title("Convergence vs diameter — kind + write pattern matter too")
fig.tight_layout()
fig.savefig(FIGS / "convergence_vs_diameter.pdf", bbox_inches="tight")
plt.show()

## Raw distributions (box plots)

Shows the full trial distribution rather than summary statistics.
More honest for small N — outliers are visible.

In [ ]:
# One figure per topology family. Each family gets its own horizontal
# budget and its own y-axis scale — partition-heal and full-mesh-n10
# differ by an order of magnitude, so a shared layout would flatten the
# smaller families against the axis. Saves as boxplot-<kind>.pdf.
for kind in sorted(trials.topology_kind.unique()):
    sub = trials[trials.topology_kind == kind]
    n_scenarios = sub.scenario.nunique()
    # Sort scenarios within this family by median convergence so the
    # fastest box is always on the left.
    order = (
        sub.groupby("scenario")["convergence_ms"]
        .median()
        .sort_values()
        .index
    )

    # Width grows with scenario count so labels never overlap, regardless
    # of family size (full_mesh has ~8 scenarios; partition_heal has 12).
    width = max(7, 0.8 * n_scenarios + 2)
    fig, ax = plt.subplots(figsize=(width, 4))
    sns.boxplot(data=sub, x="scenario", y="convergence_ms", order=order, ax=ax)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    ax.set_xlabel(None)
    ax.set_ylabel("Convergence (ms)")
    ax.set_title(f"Convergence distribution — {kind}")
    fig.tight_layout()
    fig.savefig(FIGS / f"boxplot-{kind}.pdf")
    plt.show()

## Summary table

Formatted for copy-paste into the thesis evaluation section.

In [ ]:
table = summary[["scenario", "node_count", "n_trials", "mean_ms", "p50_ms", "p95_ms"]].copy()
table.columns = ["Scenario", "Nodes", "Trials", "Mean (ms)", "p50 (ms)", "p95 (ms)"]
table = table.set_index("Scenario")
table.round(1)

## Measurement stability

Standard deviation and coefficient of variation (CV = σ/μ) across trials.
CV < 10% is generally stable; CV > 30% suggests the measurement is noisy
and more trials or a quieter environment are needed.

In [ ]:
stability = (
    trials.groupby("scenario")["convergence_ms"]
    .agg(mean="mean", std="std", n="count")
    .assign(cv_pct=lambda df: (df["std"] / df["mean"] * 100).round(1))
    .round({"mean": 2, "std": 2})
    .sort_values("cv_pct", ascending=False)
)
stability.columns = ["Mean (ms)", "Std (ms)", "N trials", "CV (%)"]

# Highlight rows where CV exceeds 20% — worth investigating
def highlight_cv(row):
    return ["background-color: #fdd" if row["CV (%)"] > 20 else "" for _ in row]

stability.style.apply(highlight_cv, axis=1)